In [1]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.getOrCreate()

In [7]:
data = spark.read.csv(
    "E:/Cluster Computing/Dataset/ca3_housing_regression_sample.csv",
    header = True,
    inferSchema = True
)

In [9]:
data.show()

+----+--------+---+-----+
|Area|Bedrooms|Age|Price|
+----+--------+---+-----+
|1200|       2| 15|  180|
|1500|       3| 10|  250|
|1800|       3|  8|  310|
|1000|       2| 20|  150|
|2200|       4|  5|  420|
|1600|       3| 12|  270|
|2000|       4|  7|  390|
|1300|       2| 18|  190|
|2500|       4|  4|  480|
|1700|       3|  9|  295|
| 900|       1| 25|  120|
|2100|       4|  6|  400|
+----+--------+---+-----+



In [10]:
data.printSchema()

root
 |-- Area: integer (nullable = true)
 |-- Bedrooms: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Price: integer (nullable = true)



In [11]:
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(
    inputCols = ["Area", "Bedrooms", "Age"],
    outputCol = "features"
)

data = assembler.transform(data)

In [12]:
data.show()

+----+--------+---+-----+-----------------+
|Area|Bedrooms|Age|Price|         features|
+----+--------+---+-----+-----------------+
|1200|       2| 15|  180|[1200.0,2.0,15.0]|
|1500|       3| 10|  250|[1500.0,3.0,10.0]|
|1800|       3|  8|  310| [1800.0,3.0,8.0]|
|1000|       2| 20|  150|[1000.0,2.0,20.0]|
|2200|       4|  5|  420| [2200.0,4.0,5.0]|
|1600|       3| 12|  270|[1600.0,3.0,12.0]|
|2000|       4|  7|  390| [2000.0,4.0,7.0]|
|1300|       2| 18|  190|[1300.0,2.0,18.0]|
|2500|       4|  4|  480| [2500.0,4.0,4.0]|
|1700|       3|  9|  295| [1700.0,3.0,9.0]|
| 900|       1| 25|  120| [900.0,1.0,25.0]|
|2100|       4|  6|  400| [2100.0,4.0,6.0]|
+----+--------+---+-----+-----------------+



In [13]:
train_data, test_data = data.randomSplit(
    [0.8, 0.2],
    seed = 42
)

In [16]:
data.count()

12

In [14]:
train_data.count()

9

In [15]:
test_data.count()

3

In [19]:
from pyspark.ml.regression import LinearRegression
regression = LinearRegression(
    featuresCol = "features",
    labelCol = "Price"
)

model = regression.fit(train_data)

In [20]:
predictions = model.transform(test_data)

In [21]:
predictions.show()

+----+--------+---+-----+-----------------+------------------+
|Area|Bedrooms|Age|Price|         features|        prediction|
+----+--------+---+-----+-----------------+------------------+
|1200|       2| 15|  180|[1200.0,2.0,15.0]| 170.1728082851604|
|1700|       3|  9|  295| [1700.0,3.0,9.0]|  289.222663776491|
|2000|       4|  7|  390| [2000.0,4.0,7.0]|379.42738439306646|
+----+--------+---+-----+-----------------+------------------+



In [24]:
from pyspark.ml.evaluation import RegressionEvaluator
metric = ['rmse', 'mae', 'r2']

for i in metric:
    evaluator = RegressionEvaluator(
        labelCol = "Price",
        predictionCol = "prediction",
        metricName = i
    )
    print(f'{i} : {evaluator.evaluate(predictions)}')

rmse : 8.976478738192318
mae : 8.72571451509405
r2 : 0.9890701652624723
